In [1]:
import os, time, math, random
from collections import defaultdict

os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['PYSPARK_PYTHON'] = os.sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = os.sys.executable

from pyspark import SparkContext, SparkConf
from pyspark.mllib.linalg import Vectors

conf = (SparkConf()
    .setAppName('assignment4')
    .setMaster('local[*]')
    .set('spark.driver.bindAddress', '127.0.0.1')
    .set('spark.driver.host', '127.0.0.1')
    .set('spark.ui.enabled', 'false'))

sc = SparkContext(conf=conf)
sc.setLogLevel('ERROR')
print('spark version:', sc.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/16 23:19:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


spark version: 4.1.1


## Part 1 - Clustering

In [2]:
def readVectorsSeq(filename):
    points = []
    with open(filename) as f:
        for line in f:
            vals = list(map(float, line.strip().split(',')))
            points.append(Vectors.dense(vals))
    return points

P = readVectorsSeq('/Users/bvijay/Downloads/MLBD_Assignment4/Assignment_4_datasets/Q1- UCI Spam clustering/spambase.data')
print(f'loaded {len(P)} points with {len(P[0])} dimensions')

loaded 4601 points with 58 dimensions


In [3]:
# helper - squared euclidean distance between two Vectors
def sqdist(a, b):
    return Vectors.squared_distance(a, b)

In [4]:
def kcenter(P, k):
    # farthest-first traversal
    # start with a random point, then greedily pick the farthest one each time
    centers = [P[random.randint(0, len(P) - 1)]]
    
    # track min distance of each point to the nearest center found so far
    min_dist = [sqdist(p, centers[0]) for p in P]

    for _ in range(k - 1):
        # next center = point with the largest min-distance
        next_idx = max(range(len(P)), key=lambda i: min_dist[i])
        centers.append(P[next_idx])
        # update min distances with the new center
        for i in range(len(P)):
            d = sqdist(P[i], P[next_idx])
            if d < min_dist[i]:
                min_dist[i] = d

    return centers

In [5]:
def kmeansPP(P, k):
    # kmeans++ seeding
    # pick first center randomly, then sample remaining ones with prob proportional to D^2
    centers = [P[random.randint(0, len(P) - 1)]]
    min_dist = [sqdist(p, centers[0]) for p in P]

    for _ in range(k - 1):
        total = sum(min_dist)
        r = random.uniform(0, total)
        cumul = 0.0
        chosen = len(P) - 1
        for i, d in enumerate(min_dist):
            cumul += d
            if cumul >= r:
                chosen = i
                break
        centers.append(P[chosen])
        for i in range(len(P)):
            d = sqdist(P[i], P[chosen])
            if d < min_dist[i]:
                min_dist[i] = d

    return centers

In [6]:
def kmeansObj(P, C):
    # average squared distance from each point to its nearest center
    total = sum(min(sqdist(p, c) for c in C) for p in P)
    return total / len(P)

In [7]:
k  = 10
k1 = 50

# step 1: run kcenter and print time
t = time.time()
C1 = kcenter(P, k)
print(f'kcenter(P, {k}) time: {time.time() - t:.3f}s')

# step 2: run kmeans++ directly on P, then compute objective
t = time.time()
C2 = kmeansPP(P, k)
print(f'kmeansPP(P, {k}) time: {time.time() - t:.3f}s')
print(f'kmeansPP objective: {kmeansObj(P, C2):.4f}')

# step 3: use kcenter to get a coreset of k1 points, then run kmeans++ on that coreset
# idea: k1 > k centers from kcenter act as a compressed representation (coreset) of P
t = time.time()
X  = kcenter(P, k1)
C3 = kmeansPP(X, k)
print(f'coreset+kmeansPP time: {time.time() - t:.3f}s')
print(f'coreset+kmeansPP objective: {kmeansObj(P, C3):.4f}')

kcenter(P, 10) time: 0.055s
kmeansPP(P, 10) time: 0.057s
kmeansPP objective: 25916.3604
coreset+kmeansPP time: 0.305s
coreset+kmeansPP objective: 90152.6026


## Part 2 - Web Search

In [8]:
# constants as given in the assignment
STOP_WORDS  = {'a','an','the','they','these','this','for','is','are','was','of','or','and','does','will','whose'}
PUNCTUATION = '{}[]<>=(). ,;\'"?#!-:'
# exhaustive list from the problem statement
PLURAL_MAP  = {'stacks':'stack','structures':'structure','applications':'application'}

In [9]:
class MySet:
    def __init__(self):
        self.items = []

    def addElement(self, element):
        if element not in self.items:
            self.items.append(element)

    def union(self, otherSet):
        result = MySet()
        for x in self.items:
            result.addElement(x)
        for x in otherSet.items:
            result.addElement(x)
        return result

    def intersection(self, otherSet):
        result = MySet()
        for x in self.items:
            if x in otherSet.items:
                result.addElement(x)
        return result

    def __len__(self):
        return len(self.items)

    def __iter__(self):
        return iter(self.items)

In [10]:
class Position:
    def __init__(self, page_entry, word_index):
        self.page_entry = page_entry
        self.word_index = word_index

    def getPageEntry(self):
        return self.page_entry

    def getWordIndex(self):
        return self.word_index

In [11]:
class WordEntry:
    def __init__(self, word):
        self.word      = word
        self.positions = []   # list of Position objects

    def addPosition(self, position):
        self.positions.append(position)

    def addPositions(self, positions):
        for p in positions:
            self.positions.append(p)

    def getAllPositionsForThisWord(self):
        return self.positions

    def getTermFrequency(self, page_name):
        # tf = count of word in this page / total words in page
        count = sum(1 for p in self.positions if p.getPageEntry().page_name == page_name)
        total = p.getPageEntry().total_words if count > 0 else 1
        return count / total

In [12]:
class MyHashTable:
    def __init__(self, size=1024):
        self.size    = size
        self.buckets = [[] for _ in range(size)]

    def getHashIndex(self, s):
        h = 0
        for ch in s:
            h = (h * 31 + ord(ch)) % self.size
        return h

    def addPositionsForWord(self, word_entry):
        idx    = self.getHashIndex(word_entry.word)
        bucket = self.buckets[idx]
        for existing in bucket:
            if existing.word == word_entry.word:
                existing.addPositions(word_entry.getAllPositionsForThisWord())
                return
        bucket.append(word_entry)

    def get(self, word):
        idx = self.getHashIndex(word)
        for entry in self.buckets[idx]:
            if entry.word == word:
                return entry
        return None

    def getWordEntries(self):
        result = []
        for bucket in self.buckets:
            result.extend(bucket)
        return result

In [13]:
class PageIndex:
    # stores one WordEntry per unique word in a single document
    def __init__(self):
        self.table = MyHashTable()

    def addPositionForWord(self, word, position):
        we = self.table.get(word)
        if we is None:
            we = WordEntry(word)
            we.addPosition(position)
            self.table.addPositionsForWord(we)
        else:
            we.addPosition(position)

    def getWordEntries(self):
        return self.table.getWordEntries()

    def getEntry(self, word):
        return self.table.get(word)

In [14]:
def clean_word(w):
    w = w.lower()
    return PLURAL_MAP.get(w, w)

def strip_punct(text):
    for ch in PUNCTUATION:
        text = text.replace(ch, ' ')
    return text


class PageEntry:
    def __init__(self, page_name, webpages_dir):
        self.page_name   = page_name
        self.page_index  = PageIndex()
        self.total_words = 0
        self._build_index(webpages_dir)

    def _build_index(self, webpages_dir):
        path = os.path.join(webpages_dir, self.page_name)
        with open(path, encoding='utf-8', errors='ignore') as f:
            text = f.read()

        # count total words (including stop words) for TF denominator
        self.total_words = len(strip_punct(text).split())

        # word index counts position including stop words and punctuation tokens
        raw_tokens = strip_punct(text).split()
        for idx, tok in enumerate(raw_tokens):
            word = clean_word(tok)
            if word and word not in STOP_WORDS:
                pos = Position(self, idx + 1)   # 1-based index
                self.page_index.addPositionForWord(word, pos)

    def getPageIndex(self):
        return self.page_index

In [15]:
class InvertedPageIndex:
    def __init__(self):
        self.global_table = MyHashTable(2048)  # word -> merged WordEntry across all pages
        self.page_entries = {}                 # page_name -> PageEntry

    def addPage(self, page_entry):
        self.page_entries[page_entry.page_name] = page_entry
        # merge every word from this page into the global index
        for we in page_entry.getPageIndex().getWordEntries():
            self.global_table.addPositionsForWord(we)

    def getPagesWhichContainWord(self, word):
        result = MySet()
        we = self.global_table.get(clean_word(word))
        if we:
            for pos in we.getAllPositionsForThisWord():
                result.addElement(pos.getPageEntry())
        return result

    def getPageEntry(self, page_name):
        return self.page_entries.get(page_name)

In [16]:
class SearchEngine:
    def __init__(self, webpages_dir):
        self.index        = InvertedPageIndex()
        self.webpages_dir = webpages_dir

    def performAction(self, action_message):
        parts  = action_message.strip().split()
        if not parts:
            return
        action = parts[0]

        if action == 'addPage':
            pe = PageEntry(parts[1], self.webpages_dir)
            self.index.addPage(pe)

        elif action == 'queryFindPagesWhichContainWord':
            word  = parts[1]
            pages = self.index.getPagesWhichContainWord(word)
            if len(pages) == 0:
                print(f'No webpage contains word {word}')
            else:
                names = sorted(pe.page_name for pe in pages)
                print(', '.join(names))

        elif action == 'queryFindPositionsOfWordInAPage':
            word, page_name = parts[1], parts[2]
            pe = self.index.getPageEntry(page_name)
            if pe is None:
                print(f'No webpage {page_name} found')
                return
            local_we = pe.getPageIndex().getEntry(clean_word(word))
            if local_we is None or len(local_we.getAllPositionsForThisWord()) == 0:
                print(f'Webpage {page_name} does not contain word {word}')
            else:
                indices = sorted(p.getWordIndex() for p in local_we.getAllPositionsForThisWord())
                print(', '.join(map(str, indices)))

In [17]:
engine = SearchEngine('/Users/bvijay/Downloads/MLBD_Assignment4/Assignment_4_datasets/Q2- webSearch/webpages/')

with open('/Users/bvijay/Downloads/MLBD_Assignment4/Assignment_4_datasets/Q2- webSearch/actions.txt') as f:
    for line in f:
        if line.strip():
            engine.performAction(line)

No webpage contains word delhi
stack_datastructure_wiki
stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines
No webpage contains word allain
stack_cprogramming
stack_cprogramming
stack_cprogramming
stack_oracle
stack_cprogramming, stack_datastructure_wiki, stackoverflow
stackmagazine


## Part 3 - PageRank

In [18]:
def load_graph(path):
    edges = set()
    nodes = set()
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            a, b = map(int, line.split())
            if a != b:  # ignore self-loops
                edges.add((a, b))
                nodes.update([a, b])
    return edges, nodes

In [19]:
def pagerank(path, beta=0.8, iters=40):
    edges, nodes = load_graph(path)
    n = len(nodes)
    print(f'{n} nodes, {len(edges)} unique edges')

    # build adjacency list - multiple edges between same pair treated as one
    adj = defaultdict(set)
    for src, dst in edges:
        adj[src].add(dst)
    for node in nodes:
        if node not in adj:
            adj[node] = set()

    # put adjacency list and initial ranks on Spark
    adj_rdd = sc.parallelize([(s, list(d)) for s, d in adj.items()])
    ranks   = sc.parallelize([(node, 1.0 / n) for node in nodes])

    teleport = (1 - beta) / n

    for _ in range(iters):
        # each node distributes its rank equally among its out-neighbors
        contribs = (
            adj_rdd.join(ranks)
            .flatMap(lambda x:
                [(dst, x[1][1] / len(x[1][0])) for dst in x[1][0]]
                if x[1][0] else []
            )
        )
        new_ranks = contribs.reduceByKey(lambda a, b: a + b).mapValues(lambda s: teleport + beta * s)

        # nodes that received no contributions still get the teleport share
        base  = sc.parallelize([(node, teleport) for node in nodes])
        ranks = base.leftOuterJoin(new_ranks).mapValues(lambda v: v[1] if v[1] is not None else v[0])

    return dict(ranks.collect())

In [20]:
# sanity check on small graph - top score should be around 0.036
small  = pagerank('/Users/bvijay/Downloads/MLBD_Assignment4/Assignment_4_datasets/graphs/small.txt')
top_sm = sorted(small.items(), key=lambda x: x[1], reverse=True)
print('top score on small graph:', round(top_sm[0][1], 3), '(expected ~0.036)')

100 nodes, 950 unique edges


top score on small graph: 0.036 (expected ~0.036)


In [21]:
scores = pagerank('/Users/bvijay/Downloads/MLBD_Assignment4/Assignment_4_datasets/graphs/whole.txt')
ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

print('top 5 nodes:')
for node, score in ranked[:5]:
    print(f'  node {node}: {score:.6f}')

print('bottom 5 nodes:')
for node, score in ranked[-5:]:
    print(f'  node {node}: {score:.6f}')

1000 nodes, 8161 unique edges


top 5 nodes:
  node 263: 0.002020
  node 537: 0.001943
  node 965: 0.001925
  node 243: 0.001853
  node 285: 0.001827
bottom 5 nodes:
  node 408: 0.000388
  node 424: 0.000355
  node 62: 0.000353
  node 93: 0.000351
  node 558: 0.000329
